# Baixar bulas do Bulario ANVISA — Paula Femina

Baixa as bulas (paciente e profissional) dos medicamentos de referencia da base estruturada
no Bulario Eletronico da ANVISA e gera um `metadata.json` por registro. Quando dois registros
tem o mesmo nome de produto (ex.: Amytril), cada um fica em uma pasta propria: `<nome>_<registro>`.

Os PDFs ficam no Drive (`Base-estruturada/BULAS_dir`); no Git ficam apenas os `metadata.json`.

**Como usar:** Ambiente de execucao > Executar tudo. No final, baixe
`bulas_paula_completo.zip` pelo icone de pasta na lateral.

In [ ]:
!pip -q install playwright
!playwright install chromium
!apt-get update -qq
!apt-get install -y -qq \
    libatk1.0-0t64 libatk-bridge2.0-0 libcups2 libxkbcommon0 \
    libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 \
    libnss3 libasound2t64 libpango-1.0-0 libcairo2

In [ ]:
import asyncio
import base64
import json
import re
import time
import unicodedata
from collections import Counter
from pathlib import Path
from urllib.parse import quote

from playwright.async_api import async_playwright

MEDICAMENTOS = [
    {"medicamento": "Metformina", "termo": "GLIFAGE", "registros": ["100890193", "100890340"]},
    {"medicamento": "Losartana potassica (25mg)", "termo": "CORUS", "registros": ["105730793"]},
    {"medicamento": "Losartana potassica (50/100mg)", "termo": "COZAAR", "registros": ["100290007"]},
    {"medicamento": "Hidroclorotiazida", "termo": "CLORANA", "registros": ["183260436"]},
    {"medicamento": "Enalapril", "termo": "RENITEC", "registros": ["100290031"]},
    {"medicamento": "Sinvastatina", "termo": "SINVASTACOR", "registros": ["100470270", "100470472"]},
    {"medicamento": "Paracetamol", "termo": "TYLENOL", "registros": ["157211214"]},
    {"medicamento": "Dipirona sodica", "termo": "NOVALGINA", "registros": ["186200018"]},
    {"medicamento": "Omeprazol", "termo": "PRATIPRAZOL", "registros": ["125680337"]},
    {"medicamento": "Amoxicilina", "termo": "AMOXIL", "registros": ["101070003"]},
    {"medicamento": "Amoxicilina (400mg/5mL, 875mg)", "termo": "NOVOCILIN", "registros": ["105730076"]},
    {"medicamento": "Salbutamol", "termo": "AEROLIN", "registros": ["101070226"]},
    {"medicamento": "Amitriptilina", "termo": "AMYTRIL", "registros": ["102980225", "102980364"]},
    {"medicamento": "Levotiroxina sodica", "termo": "PURAN T4", "registros": ["183260353"]},
    {"medicamento": "Ibuprofeno", "termo": "ADVIL", "registros": ["192900005"]},
    {"medicamento": "Acido acetilsalicilico", "termo": "ASPIRINA", "registros": ["170560020"]},
    {"medicamento": "Captopril", "termo": "CAPTOSEN", "registros": ["141070025"]},
    {"medicamento": "Insulina NPH humana", "termo": "HUMULIN N", "registros": None},
]

PASTA_SAIDA = Path("/content/BULAS_dir")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

BASE = "https://consultas.anvisa.gov.br"
API_PDF = BASE + "/api/consulta/medicamentos/arquivo/bula/parecer/"

def limpar_nome(texto):
    texto = texto.strip().lower()
    normalized = unicodedata.normalize("NFKD", texto)
    texto = normalized.encode("ascii", "ignore").decode("ascii")
    texto = re.sub(r'[\\/:*?"<>|]', "_", texto)
    texto = re.sub(r"\s+", "_", texto)
    return texto

In [ ]:
async def buscar_json_no_navegador(page, termo, tentativas=3):
    for tentativa in range(1, tentativas + 1):
        cache_buster = int(time.time() * 1000)
        url = (
            "https://consultas.anvisa.gov.br/#/bulario/q/"
            f"?nomeProduto={quote(termo)}&_={cache_buster}"
        )
        print(f"Abrindo pesquisa: {termo} (tentativa {tentativa}/{tentativas})")
        try:
            async with page.expect_response(
                lambda r: "/api/consulta/bulario?" in r.url, timeout=30000
            ) as response_info:
                await page.goto(
                    "https://consultas.anvisa.gov.br/#/bulario/",
                    wait_until="domcontentloaded", timeout=60000
                )
                await page.wait_for_timeout(1000)
                await page.goto(url, wait_until="domcontentloaded", timeout=60000)
            response = await response_info.value
            print("API:", response.status)
            if response.status == 200:
                return await response.json()
            print("Erro da API:", response.status, (await response.text())[:300])
        except Exception as e:
            print(f"Erro capturando resposta: {e}")
        if tentativa < tentativas:
            espera = tentativa * 5
            print(f"Aguardando {espera}s antes de tentar novamente...")
            await asyncio.sleep(espera)
    return None


JS_BAIXAR_PDF = """
        async (url) => {
            const response = await fetch(url, { method: "GET", credentials: "include" });
            if (!response.ok) {
                return { ok: false, status: response.status, erro: await response.text() };
            }
            const buffer = await response.arrayBuffer();
            const bytes = new Uint8Array(buffer);
            let binary = "";
            const chunk = 8192;
            for (let i = 0; i < bytes.length; i += chunk) {
                binary += String.fromCharCode(...bytes.subarray(i, i + chunk));
            }
            return { ok: true, status: response.status, base64: btoa(binary) };
        }
        """


async def baixar_pdf(page, token, destino, tentativas=3):
    url = f"{API_PDF}{token}/?Authorization="
    for tentativa in range(1, tentativas + 1):
        try:
            resultado = await page.evaluate(JS_BAIXAR_PDF, url)
        except Exception as e:
            resultado = {"ok": False, "status": f"excecao: {e}"}
        if resultado["ok"]:
            dados = base64.b64decode(resultado["base64"])
            destino.write_bytes(dados)
            print(f"   OK: {destino.name} ({len(dados)/1024:.1f} KB)")
            return True
        print(f"   ERRO PDF {destino.name} (HTTP {resultado['status']}) - tentativa {tentativa}/{tentativas}")
        if tentativa < tentativas:
            await asyncio.sleep(tentativa * 5)
    return False

In [ ]:
async def processar_medicamento(page, item):
    medicamento_display = item["medicamento"]
    termo = item["termo"]
    registros_esperados = item["registros"]

    print("\n" + "=" * 60)
    print(f"{medicamento_display} -> buscando: {termo}")
    print("=" * 60)

    dados = await buscar_json_no_navegador(page, termo)
    resultados = (dados or {}).get("content", [])
    print(f"Resultados retornados pela ANVISA: {len(resultados)}")

    if registros_esperados:
        selecionados = [
            r for r in resultados if str(r.get("numeroRegistro")) in registros_esperados
        ]
    else:
        selecionados = resultados[:1]
        if selecionados:
            print("AVISO: sem registro de referencia definido, pegando o 1o resultado.")

    if not selecionados:
        print("NENHUM resultado bateu com o(s) registro(s) esperado(s).")
        for r in resultados:
            print("  -", r.get("nomeProduto"), "| registro:", r.get("numeroRegistro"))
        return {"medicamento": medicamento_display, "termo": termo, "status": "nao_encontrado"}

    status_item = {
        "medicamento": medicamento_display, "termo": termo, "status": "ok",
        "arquivos": [], "falhas_pdf": [],
    }

    # Se dois registros selecionados tiverem o mesmo nomeProduto (ex.: AMYTRIL 10 mg,
    # registro 102980364, e AMYTRIL 25/75 mg, registro 102980225), os dois cairiam na
    # mesma pasta e o metadata.json do ultimo sobrescreveria o do primeiro.
    # Nesses casos cada registro ganha a sua propria pasta: <nome>_<registro>.
    contagem_pastas = Counter(
        limpar_nome(r.get("nomeProduto", termo)) for r in selecionados
    )

    for item_res in selecionados:
        nome = item_res.get("nomeProduto", termo)
        registro = item_res.get("numeroRegistro")
        empresa = item_res.get("razaoSocial", "empresa_desconhecida")
        print(f"\nProduto : {nome}\nRegistro: {registro}\nEmpresa : {empresa}")

        nome_pasta = limpar_nome(nome)
        if contagem_pastas[nome_pasta] > 1:
            nome_pasta = f"{nome_pasta}_{registro}"
        pasta = PASTA_SAIDA / nome_pasta
        pasta.mkdir(parents=True, exist_ok=True)

        token_paciente = item_res.get("idBulaPacienteProtegido")
        token_profissional = item_res.get("idBulaProfissionalProtegido")

        for tipo, token in (("paciente", token_paciente), ("profissional", token_profissional)):
            if not token:
                print(f"   Sem bula do {tipo} cadastrada na ANVISA para o registro {registro}")
                continue
            destino = pasta / f"{registro}_{tipo}.pdf"
            if await baixar_pdf(page, token, destino):
                status_item["arquivos"].append(destino.name)
            else:
                status_item["falhas_pdf"].append(destino.name)

        metadata = {
            "idProduto": item_res.get("idProduto"),
            "numeroRegistro": registro,
            "nomeProduto": nome,
            "expediente": item_res.get("expediente"),
            "razaoSocial": empresa,
            "cnpj": item_res.get("cnpj"),
            "numeroTransacao": item_res.get("numeroTransacao"),
            "data": item_res.get("data"),
            "numProcesso": item_res.get("numProcesso"),
            "dataAtualizacao": item_res.get("dataAtualizacao"),
        }
        with open(pasta / "metadata.json", "w", encoding="utf-8") as f:
            json.dump(metadata, f, ensure_ascii=False, indent=2)

    if status_item["falhas_pdf"]:
        status_item["status"] = "parcial"

    return status_item

In [ ]:
resultados_finais = []

async def main(lista=MEDICAMENTOS):
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True, args=["--no-sandbox", "--disable-dev-shm-usage"]
        )
        context = await browser.new_context(
            locale="pt-BR",
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/153.0.0.0 Safari/537.36"
            ),
        )
        page = await context.new_page()

        print("Abrindo o Bulario da ANVISA...")
        await page.goto(
            "https://consultas.anvisa.gov.br/#/bulario/",
            wait_until="domcontentloaded", timeout=60000
        )
        await page.wait_for_timeout(5000)

        for item in lista:
            try:
                r = await processar_medicamento(page, item)
                resultados_finais.append(r)
            except Exception as e:
                print(f"ERRO em {item['termo']}: {e}")
                resultados_finais.append({"medicamento": item["medicamento"], "termo": item["termo"], "status": "erro", "erro": str(e)})
            print("\nAguardando 5 segundos...")
            await asyncio.sleep(5)

        await browser.close()
        print("\nFinalizado!")

# Rodar todos os medicamentos:
await main(MEDICAMENTOS)

# Para reprocessar apenas o Amytril, comente a linha acima e descomente a de baixo:
# await main([m for m in MEDICAMENTOS if m["termo"] == "AMYTRIL"])


In [ ]:
print("RESUMO FINAL — base Estruturada (RENAME + Bulario), Paula Femina\n")
contagem = Counter(r["status"] for r in resultados_finais)
falharam = len(resultados_finais) - contagem["ok"] - contagem["parcial"]
print(
    f"Medicamentos: {len(resultados_finais)} | OK: {contagem['ok']} | "
    f"Parcial: {contagem['parcial']} | Falharam: {falharam}"
)
pdfs_ok = sum(len(r.get("arquivos", [])) for r in resultados_finais)
pdfs_falha = [f for r in resultados_finais for f in r.get("falhas_pdf", [])]
print(f"PDFs baixados: {pdfs_ok} | PDFs com falha: {len(pdfs_falha)}\n")

marcas = {"ok": "OK", "parcial": "PARCIAL"}
for r in resultados_finais:
    print(f"[{marcas.get(r['status'], 'FALHOU')}] {r['medicamento']} ({r['termo']}) - {r['status']}")
    for arquivo in r.get("falhas_pdf", []):
        print(f"      falhou: {arquivo}")

termos_refazer = [r["termo"] for r in resultados_finais if r["status"] != "ok"]
if termos_refazer:
    print("\nATENCAO: ha itens incompletos. Para reprocessar so eles, rode:")
    print(f"await main([m for m in MEDICAMENTOS if m['termo'] in {termos_refazer}])")

In [ ]:
import shutil
shutil.make_archive("/content/bulas_paula_completo", "zip", "/content/BULAS_dir")
print("Pronto! Baixe bulas_paula_completo.zip pelo painel de Arquivos (icone de pasta) a esquerda.")